# Colab SSH Setup for VS Code + Claude Code

Run these cells in Google Colab to:
1. Set up SSH access via cloudflared tunnel
2. Clone the repo
3. Install dependencies

Then connect from VS Code using Remote-SSH and run Claude Code in the terminal.

## Step 1: Verify GPU

In [ ]:
!nvidia-smi

## Step 2: Set up SSH via cloudflared (no account needed)

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Set root password for SSH
import subprocess
subprocess.run(['bash', '-c', 'echo root:ColabGPU123 | chpasswd'], check=True)

# Enable root SSH login
!sed -i 's/#PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/#PasswordAuthentication.*/PasswordAuthentication yes/' /etc/ssh/sshd_config
!service ssh restart

print("SSH configured. Password: ColabGPU123")

In [ ]:
# Start cloudflared tunnel (this cell keeps running — that's normal)
# Copy the URL it prints — you'll need it for VS Code
import subprocess
import re
import time

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Wait for the tunnel URL
for i in range(30):
    line = proc.stderr.readline()
    if 'trycloudflare.com' in line:
        url = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if url:
            hostname = url.group().replace('https://', '')
            print(f"\n{'='*60}")
            print(f"TUNNEL HOSTNAME: {hostname}")
            print(f"{'='*60}")
            print(f"\nNow configure VS Code (see Step 3 below)")
            print(f"Username: root")
            print(f"Password: ColabGPU123")
            break
    time.sleep(1)
else:
    print("Tunnel failed to start. Check output above.")

# Keep the tunnel alive
proc.wait()

## Step 3: VS Code SSH Config

### One-time setup on your LOCAL machine:

1. **Install cloudflared locally** (needed as SSH proxy):
   - Windows: `winget install cloudflare.cloudflared`
   - Or download from: https://github.com/cloudflare/cloudflared/releases

2. **Install VS Code extension**: `Remote - SSH` (by Microsoft)

3. **Edit your SSH config** (`~/.ssh/config` or `C:\Users\YOU\.ssh\config`):

```
Host colab
    HostName PASTE_HOSTNAME_HERE
    User root
    ProxyCommand cloudflared access ssh --hostname %h
```

Replace `PASTE_HOSTNAME_HERE` with the hostname printed above (e.g. `random-words.trycloudflare.com`)

4. **Connect**: In VS Code, press `Ctrl+Shift+P` → "Remote-SSH: Connect to Host" → select `colab`
5. **Password**: `ColabGPU123`

You're now in the Colab machine with full GPU access!

## Step 4: Setup repo + Claude Code (run in VS Code terminal after connecting)

Once connected via SSH in VS Code terminal:

```bash
# Clone repo
cd /content
git clone https://github.com/nihal-gazi/Native-Binarization.git
cd Native-Binarization/v2-cifar10

# Install Python deps
pip install torch torchvision scipy numpy

# Install Claude Code
curl -fsSL https://cli.anthropic.com/install.sh | sh

# Start Claude Code
claude
```

Now you have Claude Code running on the Colab T4 GPU machine!